In [2]:
using Graphs
using Random
using Statistics

# Optimized

In [3]:
using Graphs
using Random
using Base.Threads

function complex_voter_model_evolution(G, t, r, m0; highest=true)

    N = nv(G)
    deg = degree(G)

    # --- states as vector ---
    states = Int8.(2 .* (rand(N) .< (1+m0)/2) .- 1)

    # --- reset configuration ---
    nodes_sorted = sort(1:N, by=n->deg[n], rev=highest)
    reset_state = similar(states) #Empty array similar to states

    cutoff =  round(Int,(1+m0)*N/2)
    for (i,node) in enumerate(nodes_sorted)
        reset_state[node] = i <= cutoff ? 1 : -1
    end

    # --- precompute active edges for reset state ---
    active_edges_reset = Tuple{Int,Int}[]
    for e in edges(G)
        n1 = src(e) #Source of the edge
        n2 = dst(e) #Destination of the edge
        if reset_state[n1] != reset_state[n2]
            push!(active_edges_reset, (n1,n2)) #Appends the active edge
        end
    end

    # --- active edges vector ---
    active_edges = Tuple{Int,Int}[]

    for e in edges(G)
        n1 = src(e) #Source of the edge
        n2 = dst(e) #Destination of the edge
        if states[n1] != states[n2]
            push!(active_edges, (n1,n2)) #Appends the active edge
        end
    end

    tt = 0.0

    while tt < t

        w = 2length(active_edges) #Voter-like dynamics
        la = w + r #Total rate

        if la == 0
            break
        end

        dt = -log(rand())/la #Sample time increment
        if tt + dt > t
            break
        end

        tt += dt

        if rand()*la < w
            # voter event

            idx = rand(1:length(active_edges)) #Select a random edge
            n1,n2 = active_edges[idx] #Get the nodes it connects
            node_change = rand(Bool) ? n1 : n2 #Choose a node to change

            states[node_change] *= -1 #Change the node

            # update only neighbors of flipped node
            empty!(active_edges) #Empties the vector without deallocating

            for e in edges(G) #Recompute active edges
                n1 = src(e)
                n2 = dst(e)
                if states[n1] != states[n2]
                    push!(active_edges, (n1,n2))
                end
            end

        else
            # reset event
            copy!(states, reset_state) #Set the states to the reset solution 
                                  #(This can only be done like this in delta polarization, but in general one needs to recalculate the after reset state every time) 
            copy!(active_edges, active_edges_reset) #Use precomputed active edges
        end
        
    end

    return states
end



using Base.Threads

function dist_complex(G, t, r, m0, samples; highest=true)
    res = zeros(Float64, samples)
    progress_counter = Threads.Atomic{Int}(0)  # thread-safe counter

    @threads for i in 1:samples
        states = complex_voter_model_evolution(G, t, r, m0; highest=highest)
        res[i] = sum(states)/nv(G)

        # update progress
        n_done = atomic_add!(progress_counter, 1)
        if n_done % max(1, samples ÷ 50) == 0  # print ~every 2%
            progress = round(n_done / samples * 100; digits=1)
            print("\rProgress: $progress%")  # \r returns cursor to start of line
            flush(stdout)                     # ensure it prints immediately
        end
    end

    println("\rProgress: 100%")  # final update to 100%
    return res
end


dist_complex (generic function with 1 method)

In [4]:
# Parameters for the simulation
n_nodes = 500
k_degree = 5
G = random_regular_graph(n_nodes, k_degree)
t = 100
r = 0
m0 = 0
samples = 1000

1000

In [ ]:
using Plots
# Run the simulation and plot the histogram
data = dist_complex(G, t, r, m0, samples)
histogram(data, bins=20, xlabel="Magnetization", ylabel="Frequency", title="Magnetization Distribution")

## Simulation: Extracting $s_{4,2}$ on a Random Regular Graph (RRG)

We will run simulations on a Random Regular Graph (RRG) with degree $k=4$ and $N=1000$ nodes. For different initial values of $m_0$ (number of nodes in state 1), we will extract the fraction $s_{4,2}$, which is the fraction of nodes with degree 4, 2 neighbors in state 1, and themselves in state 1.

In [5]:
using Graphs

# Parameters
k = 4
N = 1000
m0_values = collect(-1:0.2:1)  # Vector of m0 values


# Placeholder for get_fraction_at_state_k_m
function get_fraction_at_state_k_m(states, G, k, m, state)
    count = 0
    total = 0
    for v in 1:N
        if degree(G, v) == k && states[v] == state
            neighbors = all_neighbors(G, v)
            m_count = sum(states[neighbors] .== 1)
            if m_count == m
                count += 1
            end
            total += 1
        end
    end
    return total == 0 ? 0.0 : count / total
end

function get_fraction_at_state_k_m_ensemble(G, m_0, r, k, m, state; ensemble_size=50)
    fractions = Float64[]
    for i in 1:ensemble_size
        # Run a simulation to get states
        states = complex_voter_model_evolution(G, 500, r, m0)
        # Compute fraction for this simulation
        fraction = get_fraction_at_state_k_m(states, G, k, m, state)
        push!(fractions, fraction)
    end
    return mean(fractions)
end

get_fraction_at_state_k_m_ensemble (generic function with 1 method)

In [ ]:
results = zeros(length(m0_values))  # Vector for results

G = random_regular_graph(N, k)
for (i, m0) in enumerate(m0_values) 
    results[i] = get_fraction_at_state_k_m_ensemble(G, m0, 0, 4, 2, 1; ensemble_size=50)
    println("$m0\t$(results[i])")
end

# Time Evolution

In [6]:
function fraction_evolution_time(G, t, r, m0, k, m, state; highest=true, every = 10)

    N = nv(G)
    deg = degree(G)
    n = 0
    evolution = Vector{Float64}()
    times = Vector{Float64}()

    # --- states as vector ---
    states = Int8.(2 .* (rand(N) .< (1+m0)/2) .- 1)

    # --- reset configuration ---
    nodes_sorted = sort(1:N, by=n->deg[n], rev=highest)
    reset_state = similar(states) #Empty array similar to states

    cutoff =  round(Int,(1+m0)*N/2)
    for (i,node) in enumerate(nodes_sorted)
        reset_state[node] = i <= cutoff ? 1 : -1
    end

    # --- precompute active edges for reset state ---
    active_edges_reset = Tuple{Int,Int}[]
    for e in edges(G)
        n1 = src(e) #Source of the edge
        n2 = dst(e) #Destination of the edge
        if reset_state[n1] != reset_state[n2]
            push!(active_edges_reset, (n1,n2)) #Appends the active edge
        end
    end

    # --- active edges vector ---
    active_edges = Tuple{Int,Int}[]

    for e in edges(G)
        n1 = src(e) #Source of the edge
        n2 = dst(e) #Destination of the edge
        if states[n1] != states[n2]
            push!(active_edges, (n1,n2)) #Appends the active edge
        end
    end

    tt = 0.0

    while tt < t
        n += 1

        w = 2length(active_edges) #Voter-like dynamics
        la = w + r #Total rate

        if la == 0
            break
        end

        dt = -log(rand())/la #Sample time increment
        if tt + dt > t
            break
        end

        tt += dt

        if rand()*la < w
            # voter event

            idx = rand(1:length(active_edges)) #Select a random edge
            n1,n2 = active_edges[idx] #Get the nodes it connects
            node_change = rand(Bool) ? n1 : n2 #Choose a node to change

            states[node_change] *= -1 #Change the node

            # update only neighbors of flipped node
            empty!(active_edges) #Empties the vector without deallocating

            for e in edges(G) #Recompute active edges
                n1 = src(e)
                n2 = dst(e)
                if states[n1] != states[n2]
                    push!(active_edges, (n1,n2))
                end
            end

        else
            # reset event
            copy!(states, reset_state) #Set the states to the reset solution 
                                  #(This can only be done like this in delta polarization, but in general one needs to recalculate the after reset state every time) 
            copy!(active_edges, active_edges_reset) #Use precomputed active edges
        end
        if n % every == 0
             push!(evolution, get_fraction_at_state_k_m(states, G, k, m, state))
             push!(times, tt)
        end
    end

    return times, evolution
end

fraction_evolution_time (generic function with 1 method)

In [9]:
using Plots
using Interpolations
using Statistics
using Base.Threads


function average_evolution_over_realizations(G, t, r, m0, k, m, state, num_realizations; highest=true, every=10)
    
    # Run all realizations and collect times and evolutions using multithreading
    all_times = Vector{Float64}[]
    all_evolutions = Vector{Float64}[]
    
    # Pre-allocate vectors for thread-safe collection
    times_array = [Vector{Float64}() for _ in 1:num_realizations]
    evolutions_array = [Vector{Float64}() for _ in 1:num_realizations]
    
    # Run realizations in parallel
    @threads for i in 1:num_realizations
        times, evolution = fraction_evolution_time(G, t, r, m0, k, m, state; highest=highest, every=every)
        times_array[i] = times
        evolutions_array[i] = evolution
    end
    
    all_times = times_array
    all_evolutions = evolutions_array
    
    # Create a common time grid from 0 to t_max
    # Use 1000 points (adjust as needed)
    t_max = maximum(maximum.(all_times))
    time_grid = range(0, t_max, length=1000)
    
    # Interpolate each realization to the common time grid
    interpolated_evolutions = Vector{Float64}[]
    
    # Pre-allocate for thread safety
    interpolated_evolutions = [Vector{Float64}(undef, length(time_grid)) for _ in 1:num_realizations]
    
    @threads for idx in 1:num_realizations
        times = all_times[idx]
        evolution = all_evolutions[idx]
        
        if length(times) > 1
            # Create interpolation function (linear interpolation)
            itp = linear_interpolation(times, evolution, extrapolation_bc=Flat())
            interp_values = itp.(time_grid)
        else
            # If only one point, repeat it
            interp_values = fill(evolution[1], length(time_grid))
        end
        interpolated_evolutions[idx] = interp_values
    end
    
    # Compute mean and std at each time point
    mean_evolution = zeros(length(time_grid))
    std_evolution = zeros(length(time_grid))
    
    for i in eachindex(time_grid)
        values_at_time = [interpolated_evolutions[j][i] for j in 1:num_realizations]
        mean_evolution[i] = mean(values_at_time)
        std_evolution[i] = std(values_at_time)
    end
    
    return time_grid, mean_evolution, std_evolution, all_times, all_evolutions
end


function plot_averaged_evolution(G, t, r, m0, k, m, state, num_realizations; highest=true, every=10, show_shaded_std=false)
    
    time_grid, mean_evolution, std_evolution, _, _ = average_evolution_over_realizations(
        G, t, r, m0, k, m, state, num_realizations; highest=highest, every=every
    )
    
    p = plot(time_grid, mean_evolution, 
             label="Mean evolution",
             xlabel="Time",
             ylabel="Fraction",
             linewidth=2,
             legend=:best)
    
    if show_shaded_std
        # Add shaded region for ±1 standard deviation
        upper_bound = mean_evolution .+ std_evolution
        lower_bound = mean_evolution .- std_evolution
        
        plot!(p, time_grid, upper_bound, 
              fillrange=lower_bound, 
              alpha=0.3, 
              label="±1 std",
              linewidth=0)
    end
    
    return p
end

plot_averaged_evolution (generic function with 1 method)

In [10]:
N = 500  # Number of nodes
k = 4
G = random_regular_graph(N, k)
t = 750
r = 0
m0 = 0
k_param, m_param, state = 4, 2, 1
num_realizations = 100  # Number of realizations to average over


time_grid, mean_evo, std_evo, all_t, all_evo = average_evolution_over_realizations(
    G, t, r, m0, k_param, m_param, state, num_realizations; highest=true, every=50
)

p = plot_averaged_evolution(G, t, r, m0, k_param, m_param, state, num_realizations)
savefig(p, "averaged_evolution.png")

"C:\\Users\\gerar\\Desktop\\Escola\\Universitat\\Master\\TFM\\voter-model-stochastic-resetting\\notebooks\\averaged_evolution.png"